In [ ]:
# !uv add arviz
# !uv add pymc
# !uv add netCDF4
# !uv add pyarrow

### if there are problems with the pymce Model widget use
#!uv remove  ipywidgets jupyterlab_widgets -y
#!uv add ipywidgets>=8.0.0 jupyterlab_widgets

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib


import pymc as pm
import arviz as az
from aml_working_paper.data.preprocess import preprocess_data
from aml_working_paper.models.bayes_hmm import build_hmm_model
from aml_working_paper.models.labelswitch import labelswitch
from aml_working_paper.routing.routing import *
from aml_working_paper.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR


### 1. Get the synthetic data and preprocess it

In [2]:
df_aml = pd.read_csv("{}/synthetic_df_aml.csv".format(RAW_DATA_DIR))

In [3]:
train_df, val_df, test_df, train_sequences, val_sequences, test_sequences, scaler = preprocess_data(df_aml)

### 2. Build the HMM 

In [5]:
model = build_hmm_model(train_sequences)

### 3. Train it

#### 3.1 Simple quick model for demonstration purposes

In [ ]:
### The model configuration should take less than a minute to run in a 8GB CPU and 16RAM computer
### If it takes more, please restart the notebook. You should also be able to see the Pymc widget of the model running
### If not, please go to the top of this notebook and run the respective commands
with model:
    trace_test = pm.sample(draws=5, tune=5, chains=4, cores=4, target_accept=0.9, random_seed=42)

Only 5 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [pi, current, mu, sigma]


Output()

Sampling 4 chains for 5 tune and 5 draw iterations (20 + 20 draws total) took 1 seconds.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
The number of samples is too small to check convergence reliably.


In [ ]:
print(az.summary(trace_test, var_names=["pi", "current", "mu", "sigma"], round_to=3 ) )

                mean     sd  eti89_lb  eti89_ub  ess_bulk  ess_tail  r_hat  \
pi[0]          0.304  0.198     0.097     0.539    19.266    19.266  3.252   
pi[1]          0.295  0.193     0.102     0.576    19.266    19.266  3.536   
pi[2]          0.402  0.107     0.324     0.579    19.266    19.266  3.369   
current[0, 0]  0.419  0.244     0.149     0.793    19.266    19.266  3.258   
current[0, 1]  0.379  0.201     0.117     0.653    19.266    19.266  4.740   
current[0, 2]  0.202  0.152     0.089     0.487    19.266    19.266  4.957   
current[1, 0]  0.228  0.125     0.114     0.385    19.266    19.266  2.581   
current[1, 1]  0.484  0.182     0.303     0.713    19.266    19.266  3.252   
current[1, 2]  0.288  0.190     0.051     0.582    19.266    19.266  3.632   
current[2, 0]  0.144  0.117     0.064     0.407    19.266    19.266  2.312   
current[2, 1]  0.190  0.220     0.034     0.593    19.266    19.266  4.486   
current[2, 2]  0.666  0.267     0.286     0.879    19.266    19.

#### 3.2 Different configuration models

In [ ]:
with model:
    trace_test = pm.sample(draws=100, tune=100, chains=4, cores=4, target_accept=0.9, random_seed=42 )

#### 3.3 Metrics

In [ ]:
print(az.summary(trace_test, var_names=["pi", "current", "mu", "sigma"], round_to=3 ) )

In [ ]:
print("Divergences:",trace_test.sample_stats["diverging"].sum().item() )

In [ ]:
print("\nBFMI:", az.bfmi(trace_test) )

In [ ]:
az.plot_dist(trace_test, var_names=["mu"])

In [ ]:
aligned_trace, permutations = labelswitch(trace_test)

In [ ]:
print(az.summary(aligned_trace,var_names=["pi", "current", "mu", "sigma"],round_to=3))

In [ ]:
az.plot_trace(aligned_trace,var_names=["mu"])

### 4. Save important data

In [ ]:
np.save("{}/train_account_ids.npy".format(PROCESSED_DATA_DIR), train_df["account_id"].unique())

np.save("{}/val_account_ids.npy".format(PROCESSED_DATA_DIR), val_df["account_id"].unique())

np.save( "{}/test_account_ids.npy".format(PROCESSED_DATA_DIR), test_df["account_id"].unique())

In [ ]:
joblib.dump(scaler, "{}/hmm_scaler.joblib".format(MODELS_DIR))